# How GrAF works

The [Introduction](Introduction.ipynb) covers saving and loading. This page
explains the *structure* inside a `.graf` file: why axes are called `Ax0`,
traces `Tr0`, and how subplots are recorded.

Understanding this is what lets you pull data out of a file without needing
the script that made it — which is the point of the format. The full
field-by-field definition is in the [format specification](../format.md).


## The shape of a file

A GrAF file is a tree:

```
Graf
├── info          metadata and provenance
├── style         fonts
└── axes          a named collection of Axis objects
    └── Ax0
        ├── x_axis, y_axis_L, y_axis_R, z_axis   the scales
        ├── traces      a named collection of Trace objects
        │   └── Tr0     x_data, y_data, colour, line type, ...
        └── surfaces    images and 3-D surfaces
```

Axes and traces are stored in **named collections**, not lists. `Ax0`, `Tr0`
and so on are keys. That is deliberate: a name is stable and readable in any
language, and it survives round-tripping through the underlying container
without depending on ordering. A reader in MATLAB or C++ can address
`axes/Ax0/traces/Tr0/x_data` directly.


## Subplots

Each `Axis` records where it sits in the figure's grid, so a subplot layout
can be rebuilt without storing matplotlib's own layout objects.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from graf.base import Graf

x = np.linspace(0, 10, 50)
fig, axs = plt.subplots(2, 2, figsize=(8, 5))

for ax, fn in zip(axs.flat, [np.sin, np.cos, np.tanh, np.sqrt]):
    ax.plot(x, fn(x), label=fn.__name__)
    ax.set_title(fn.__name__)

fig.tight_layout()
graf = Graf(fig)
plt.show()


Four subplots become four entries in `axes`:


In [ ]:
for key, axis in graf.axes.items():
    print(f'{key}: position={axis.position}  span={axis.span}  traces={list(axis.traces.keys())}')


`position` is `[row, column]` counting from the top-left, and `span` is how
many rows and columns the axes occupies — so an axes created with
`subplot2grid` spanning two columns records `span=[1, 2]`.

Note that every subplot has its own `Tr0`. Trace names are scoped to their
axes, so the first trace on each subplot is `Tr0`; they do not run
continuously across the figure.


## Getting at the data

Because the names are predictable, reading a specific series is a matter of
indexing two dictionaries:


In [ ]:
trace = graf.axes['Ax1'].traces['Tr0']

print('label :', trace.display_name)
print('x[:5] :', trace.x_data[:5])
print('y[:5] :', trace.y_data[:5])
print('style :', trace.line_type, '| colour:', trace.line_color)


The data is a plain list of floats. There is no object graph to reconstruct
and no matplotlib version to match — that is the property GrAF exists to
preserve.


## The four scales

Every `Axis` carries four `Scale` objects: `x_axis`, `y_axis_L`, `y_axis_R`
and `z_axis`. Unused ones are still present, marked invalid, which keeps the
structure of every file identical and means a reader never has to test
whether a field exists.


In [ ]:
axis = graf.axes['Ax0']
for name in ('x_axis', 'y_axis_L', 'y_axis_R', 'z_axis'):
    scale = getattr(axis, name)
    print(f'{name:9s} valid={scale.is_valid}')


### Twin axes

A second y-axis is not a second `Axis`. It is the same `Axis` with
`y_axis_R` marked valid, and each trace saying which side it belongs to via
`use_yaxis_R`. Keeping a twinned pair together is what preserves the fact
that the two series share an x-axis.


In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 3))
ax2.plot(x, np.sin(x), color='tab:blue', label='left')
ax2.twinx().plot(x, 100 * np.cos(x), color='tab:red', label='right')

twin = Graf(fig2).axes['Ax0']
print('y_axis_R valid:', twin.y_axis_R.is_valid)
for key, tr in twin.traces.items():
    side = 'right' if tr.use_yaxis_R else 'left'
    print(f'  {key}: {side}')
plt.show()


## Where to go next

- The [format specification](../format.md) defines every field on disk,
  including the versioning rules a reader must follow.
- The [API reference](../api/index.rst) documents these classes directly.
